In [ ]:
# Cell 1 — Workspace setup (RUN FIRST)
from pathlib import Path

BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Compatibility aliases for notebook examples
RAW = RAW_DIR
PROCESSED = PROCESSED_DIR
BASE = BASE_DIR
OUT = PROCESSED_DIR

print("Raw folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)


Raw folder: /content/week3_project/data/raw
Processed folder: /content/week3_project/data/processed


In [ ]:
# Cell 3 — Verify uploaded inputs (self-contained)
from pathlib import Path
RAW = Path("/content/week3_project/data/raw")
RAW_DIR = RAW
BASE_DIR = RAW.parent.parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
BASE = BASE_DIR
OUT = PROCESSED_DIR

files = sorted(f for f in RAW.rglob("*") if f.is_file())
print("Input count:", len(files))
for f in files:
 	print(f.relative_to(RAW), "bytes:", f.stat().st_size)

assert files, "Upload at least one relevant input file before proceeding."
print("Upload verified. Select your relevant input(s) for Step A3.")


Input count: 5
utm_User-Manual_Course-Registration-Senior-Student_update202620271.pdf bytes: 2055792
utm_User-Manual_NewStudent_Course-Registration_update-202620271.pdf bytes: 2237349
utm_academic_calendar_2026_2027.pdf bytes: 627618
utm_academic_regulations.pdf bytes: 779259
utm_handbook.pdf bytes: 516792
Upload verified. Select your relevant input(s) for Step A3.


In [ ]:
# Cell 4 — Inspect source documents (A3)
!pip -q install pypdf
import re, csv
from pypdf import PdfReader

summary = []

for f in sorted(RAW.rglob("*.pdf")):
    reader = PdfReader(str(f))
    texts = [(p.extract_text() or "") for p in reader.pages]
    chars = sum(len(t) for t in texts)
    low = [i + 1 for i, t in enumerate(texts) if len(t.strip()) < 20]
    gid = sum(len(re.findall(r"/gid\d+", t)) for t in texts)
    meta = reader.metadata or {}

    print("=" * 70)
    print("FILE:", f.name, "| bytes:", f.stat().st_size)
    print("Pages:", len(texts), "| characters:", chars)
    print("Pages with little/no text:", low)
    print("Garbled glyph codes (/gid...):", gid)
    print("Created:", meta.get("/CreationDate"), "| Modified:", meta.get("/ModDate"))
    sample = next((t for t in texts if t.strip()), "")
    print("--- Sample (first 400 chars of first non-empty page) ---")
    print(sample[:400])

    summary.append({
        "file": f.name, "bytes": f.stat().st_size, "pages": len(texts),
        "chars": chars, "low_text_pages": low, "gid_tokens": gid,
    })

# Save a summary table as evidence
out = PROCESSED_DIR / "inspection_summary.csv"
with open(out, "w", newline="", encoding="utf-8") as fh:
    w = csv.DictWriter(fh, fieldnames=summary[0].keys())
    w.writeheader()
    w.writerows(summary)
print("\nSaved:", out)

FILE: utm_User-Manual_Course-Registration-Senior-Student_update202620271.pdf | bytes: 2055792
Pages: 10 | characters: 2048
Pages with little/no text: []
Garbled glyph codes (/gid...): 0
Created: D:20261006095121+08'00' | Modified: D:20261006113042+08'00'
--- Sample (first 400 chars of first non-empty page) ---
Step-by-step
COURSE 
REGISTRATION 
Bachelor Degree (Fulltime) 
Senior Student

FILE: utm_User-Manual_NewStudent_Course-Registration_update-202620271.pdf | bytes: 2237349
Pages: 24 | characters: 3752
Pages with little/no text: [1, 2, 5]
Garbled glyph codes (/gid...): 0
Created: D:20261005111036+08'00' | Modified: D:20261006112023+08'00'
--- Sample (first 400 chars of first non-empty page) ---
www.utm.my

FILE: utm_academic_calendar_2026_2027.pdf | bytes: 627618
Pages: 2 | characters: 8335
Pages with little/no text: []
Garbled glyph codes (/gid...): 0
Created: D:20260512100557+08'00' | Modified: D:20260513190350+08'00'
--- Sample (first 400 chars of first non-empty page) ---
DATE P

In [ ]:
# Cell 4b — Readable evidence pages for the inventory
def show(filename_part, pages, n=500):
    f = next(f for f in RAW.rglob("*.pdf") if filename_part in f.name)
    r = PdfReader(str(f))
    for p in pages:
        t = (r.pages[p - 1].extract_text() or "")
        print(f"--- {f.name} | page {p} ---")
        print(t[:n], "\n")

show("academic_calendar", [1])
show("academic_regulations", [20, 40])

--- utm_academic_calendar_2026_2027.pdf | page 1 ---
DATE PROGRAMME PUBLIC HOLIDAY 
International Undergraduate and 
Postgraduate Students: 
21 - 24 September 2026 (UTM JB) 
26 & 27 September 2026 (UTM KL)
Local Undergraduate Students:      
4 October 2026 (UTM JB & UTM KL)  
Local Postgraduate Students: 
14 -18 September 2026 (UTM JB) 
26 & 27 September 2026 (UTM KL)
Registration of New Students 
Semester I 2026/2027 Session 
(September Intake) 
21 September - 2 October 2026 International Student Orientation Programme  (ISOP) 
4 October 2026 - 9  

--- utm_academic_regulations.pdf | page 20 ---
3
Academic Regulations Section 1
“National Interest” refers to a declaration by the government as a matter 
of  national interest.
“Graduation Credits”  refers to the total credits required for a student to 
graduate, in accordance with the prescribed curriculum code.
“Earned Credits” refers to the total credits acquired by a student for 
courses that have been passed.
“Calculated Credits” refe

In [ ]:
# Cell 4 — REQUIRED: Source Selection + Lineage Utilities
# Run ONCE after Cell 1 and Cell 3

from pathlib import Path
import json
import hashlib
from datetime import datetime, timezone

# Select the exact source before parsing.
# Never automatically take the first file by extension.

def select_source(relative_path=None, filename=None, suffixes=None):
    if relative_path:
        selected = RAW_DIR / relative_path
        if not selected.is_file():
            raise FileNotFoundError(
                f"Source not found: {selected}"
            )
        return selected
    matches = sorted(
        p for p in RAW_DIR.rglob("*")
        if p.is_file()
        and (filename is None or p.name == filename)
        and (suffixes is None or p.suffix.lower() in suffixes)
    )
    print(
        "Candidates:",
        [str(p.relative_to(RAW_DIR)) for p in matches]
    )
    if len(matches) != 1:
        raise ValueError(
            "Select exactly one source. "
            "Set relative_path to an exact file under RAW_DIR."
        )
    return matches[0]
def source_hash(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()
def save_manifest(records):
    manifest = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "records": records
    }
    target = BASE_DIR / "data" / "week3_manifest.json"
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(
        json.dumps(
            manifest,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )
    print("Saved lineage:", target)
    return manifest

# Confirmation messages
print("Cell 4 completed successfully.")
print("Source selection and lineage utilities are ready.")
print("Available functions:")
print("- select_source()")
print("- source_hash()")
print("- save_manifest()")


Cell 4 completed successfully.
Source selection and lineage utilities are ready.
Available functions:
- select_source()
- source_hash()
- save_manifest()


In [ ]:
# CELL 4A — COMMON LINEAGE REGISTRY
# REQUIRED: Run once before selected Part B pathways

from pathlib import Path
lineage_entries = {}
def register_lineage(source, output, processing):
    source = Path(source).resolve()
    output = Path(output).resolve()
    # Validate source
    if not source.is_file():
        raise FileNotFoundError(
            f"Source file not found: {source}"
        )
    # Validate prepared output
    if not output.is_file():
        raise FileNotFoundError(
            f"Prepared output not found: {output}"
        )
    if output.stat().st_size == 0:
        raise ValueError(
            f"Prepared output is empty: {output}"
        )
    # Validate processing description
    if not isinstance(processing, str) or not processing.strip():
        raise ValueError(
            "A processing description is required."
        )
    processing = processing.strip()
    if processing.upper().startswith("REPLACE"):
        raise ValueError(
            "Replace the placeholder with the actual method."
        )
    # Ensure source and output belong to project workspace
    source.relative_to(RAW_DIR.resolve())
    output.relative_to(PROCESSED_DIR.resolve())
    # Same source/output pair is updated, not duplicated
    key = (str(source), str(output))
    lineage_entries[key] = {
        "source": source,
        "output": output,
        "processing": processing
    }
    print(
        "Registered:",
        source.name,
        "->",
        output.name
    )
print("Common lineage registry ready.")
print("Registered entries:", len(lineage_entries))


Common lineage registry ready.
Registered entries: 0


In [ ]:
# Cell 5A0 — Source configuration

from pathlib import Path
import re, json

SOURCES = [
    dict(source_id="S01", match="academic_calendar", language_default="en",
         language_by_page={1: "en", 2: "ms"},
         doc_date="2026-05-14", edition="Academic Calendar 2026/2027 (corrected version)",
         currency="current_2026", page_offset=0,
         drop=[r"^www\.utm\.my$"], splitter="calendar", size=1200,
         note="Page 2 (Malay) has date errors; treat page 1 (English) as primary."),
    dict(source_id="S02", match="academic_regulations", language_default="en",
         language_by_page={},
         doc_date="2026", edition="Academic Regulations, Full-Time UG, 16th Edition",
         currency="current_2026", page_offset=17,
         drop=[], splitter="paragraph", size=800,
         note="PDF page = printed page + 17."),
    dict(source_id="S03", match="handbook", language_default="en",
         language_by_page={},
         doc_date="2010", edition="Academic Advising: Ensuring Student Success (2010)",
         currency="currency_not_verified", page_offset=0,
         drop=[r"^\d+ \| Academic Advising: Ensuring Student Success$"],
         splitter="paragraph", size=800,
         note="2010 booklet; describes older systems (AIMS 2000)."),
    dict(source_id="S04", match="NewStudent", language_default="en",
         language_by_page={},
         doc_date="2026-10-05", edition="New-student registration manual (update 2026/2027)",
         currency="currency_not_verified", page_offset=0,
         drop=[r"^www\.utm\.my$"], splitter="paragraph", size=1200,
         note="Slide deck; screenshots (image-only) show 2024 sessions."),
    dict(source_id="S05", match="Senior-Student", language_default="en",
         language_by_page={},
         doc_date="2026-10-06", edition="Senior-student registration manual (update 2026/2027)",
         currency="currency_not_verified", page_offset=0,
         drop=[r"^www\.utm\.my$"], splitter="paragraph", size=1200,
         note="Slide deck; tables and slip are image-only; screenshots show 2024 sessions."),
]

for s in SOURCES:
    matches = [p for p in RAW_DIR.rglob("*.pdf") if s["match"] in p.name]
    assert len(matches) == 1, f"{s['source_id']}: expected 1 file for {s['match']!r}, found {matches}"
    s["path"] = matches[0]
    print(s["source_id"], "->", s["path"].name)

S01 -> utm_academic_calendar_2026_2027.pdf
S02 -> utm_academic_regulations.pdf
S03 -> utm_handbook.pdf
S04 -> utm_User-Manual_NewStudent_Course-Registration_update-202620271.pdf
S05 -> utm_User-Manual_Course-Registration-Senior-Student_update202620271.pdf


In [ ]:
# Cell 5A1 — Extract, clean, chunk (two approaches compared)

!pip -q install pypdf
from pypdf import PdfReader

MIN_PAGE_CHARS = 20   # pages with less text than this are treated as image-only / blank

def clean_page(text, drop_patterns):
    text = re.sub(r"/gid\d+", "", text)               # glyph-code noise (S02 title page)
    lines = [ln.rstrip() for ln in text.splitlines()]
    keep = []
    for ln in lines:
        if any(re.match(p, ln.strip()) for p in drop_patterns):
            continue                                    # repeated footer / running line
        keep.append(ln)
    text = "\n".join(keep)
    text = re.sub(r"[ \t]+", " ", text)                 # collapse spaces
    text = re.sub(r"\n{3,}", "\n\n", text)              # collapse blank-line runs
    return text.strip()

def split_units(text):
    """Paragraphs; very long paragraphs are split at sentence/line ends."""
    units = []
    for para in re.split(r"\n\s*\n", text):
        para = para.strip()
        if not para:
            continue
        units.append(para)
    return units

def pack(units, size):
    chunks, cur = [], ""
    for u in units:
        if len(u) > size:                               # split long unit at sentence/line ends
            if cur:
                chunks.append(cur); cur = ""
            pieces = re.split(r"(?<=[.!?;:])\s+|\n", u)
            buf = ""
            for pc in pieces:
                if len(buf) + len(pc) + 1 <= size:
                    buf += (" " if buf else "") + pc
                else:
                    if buf: chunks.append(buf)
                    buf = pc[:size] if len(pc) > size else pc
            if buf: chunks.append(buf)
        elif len(cur) + len(u) + (2 if cur else 0) <= size:
            cur += ("\n\n" if cur else "") + u
        else:
            chunks.append(cur); cur = u
    if cur:
        chunks.append(cur)
    return chunks

def calendar_chunks(text, size):
    parts = re.split(r"(?m)^(?=SEMESTER I \||SEMESTER II \||SHORT SEMESTER \|)", text)
    out = []
    for p in parts:
        p = p.strip()
        if p:
            out.append(p)       # keep each semester block whole so rows stay together
    return out if len(out) > 1 else pack(split_units(text), size)

def parse_printed_page(text):
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    for cand in ([lines[0], lines[-1]] if lines else []):
        if re.fullmatch(r"\d{1,3}", cand):
            return int(cand), cand
    return None, None

def section_of(text):
    for l in [l.strip() for l in text.splitlines() if l.strip()][:3]:
        m = re.search(r"Section\s+(\d+)", l)
        if m:
            return f"Section {m.group(1)}"
    return None

prepared_records, evidence, mismatch_notes = [], [], []

for s in SOURCES:
    reader = PdfReader(str(s["path"]))
    s["sha256"] = source_hash(s["path"])
    pages_clean, skipped = [], []
    last_section = None
    for pno, page in enumerate(reader.pages, 1):
        raw = page.extract_text() or ""
        printed, printed_line, inferred = None, None, False
        if s["page_offset"]:
            printed, printed_line = parse_printed_page(raw)
            if printed is not None and pno >= 18 and printed != pno - s["page_offset"]:
                mismatch_notes.append((s["source_id"], pno, printed))
            if printed is None and pno >= 18:           # number line not found: infer from offset
                printed, inferred = pno - s["page_offset"], True
            sec = section_of(raw)
            if sec: last_section = sec
            if printed_line:                            # drop the bare page-number line and running header
                raw = "\n".join(l for l in raw.splitlines() if l.strip() != printed_line)
            raw = "\n".join(l for l in raw.splitlines()
                            if not re.fullmatch(r"\s*(Academic Regulations\s+)?Section\s+\d+\s*", l))
        text = clean_page(raw, s["drop"])
        if len(text) < MIN_PAGE_CHARS:
            skipped.append(pno)
            continue
        pages_clean.append((pno, text, printed, last_section, inferred))

    # ---- Approach B (selected): page-aware, structure-preserving ----
    count_B = 0
    for pno, text, printed, sec, inferred in pages_clean:
        if s["splitter"] == "calendar" and pno == 1:
            chunks = calendar_chunks(text, s["size"])
        else:
            chunks = pack(split_units(text), s["size"])
        lang = s["language_by_page"].get(pno, s["language_default"])
        for k, ch in enumerate(chunks, 1):
            prepared_records.append({
                "record_id": f"{s['source_id']}-p{pno:03d}-{k:02d}",
                "source_id": s["source_id"],
                "source": str(s["path"].relative_to(RAW_DIR)),
                "source_sha256": s["sha256"],
                "edition": s["edition"],
                "doc_date": s["doc_date"],
                "currency_status": s["currency"],
                "language": lang,
                "pdf_page": pno,
                "printed_page": printed,
                "printed_page_inferred": inferred,
                "section": sec,
                "reliability_note": (s["note"] if s["source_id"] != "S01" or pno == 2 else None),
                "processing": f"page-aware; paragraph/sentence packing; max_chars={s['size']}; footers removed; whitespace normalised",
                "text": ch,
            })
            count_B += 1

    # ---- Approach A (baseline, lab default): fixed 500 chars over whole document ----
    joined, boundaries = "", []
    for pno, text, _, _, _ in pages_clean:
        boundaries.append(len(joined)); joined += text + "\n"
    A = [(i, joined[i:i + 500]) for i in range(0, len(joined), 500)]
    A_cross = sum(1 for i, c in A if any(i < b < i + len(c) for b in boundaries[1:]))
    A_mid = sum(1 for i, c in A if c[:1].islower())
    B_recs = [r for r in prepared_records if r["source_id"] == s["source_id"]]
    B_mid = sum(1 for r in B_recs if r["text"][:1].islower())
    evidence.append(dict(source=s["source_id"], pages=len(reader.pages), skipped_pages=len(skipped),
                         A_chunks=len(A), A_cross_page=A_cross, A_start_lowercase=A_mid,
                         B_chunks=len(B_recs), B_cross_page=0, B_start_lowercase=B_mid))
    s["skipped_pages"] = skipped

print("Printed-page mismatches vs offset 17 (inspect these):", mismatch_notes[:10])
print("\n--- Approach A (fixed 500 chars, whole document) vs B (page-aware) ---")
for e in evidence:
    print(e)
print("\nTotal prepared records:", len(prepared_records))
print("Skipped image-only/blank pages:", {s["source_id"]: s["skipped_pages"] for s in SOURCES})
print("\nSample S02 record:\n", json.dumps(next(r for r in prepared_records if r["source_id"] == "S02" and r["pdf_page"] >= 20), indent=2)[:900])

Printed-page mismatches vs offset 17 (inspect these): []

--- Approach A (fixed 500 chars, whole document) vs B (page-aware) ---
{'source': 'S01', 'pages': 2, 'skipped_pages': 0, 'A_chunks': 17, 'A_cross_page': 1, 'A_start_lowercase': 9, 'B_chunks': 8, 'B_cross_page': 0, 'B_start_lowercase': 0}
{'source': 'S02', 'pages': 81, 'skipped_pages': 7, 'A_chunks': 205, 'A_cross_page': 71, 'A_start_lowercase': 164, 'B_chunks': 176, 'B_cross_page': 0, 'B_start_lowercase': 50}
{'source': 'S03', 'pages': 28, 'skipped_pages': 2, 'A_chunks': 73, 'A_cross_page': 24, 'A_start_lowercase': 56, 'B_chunks': 69, 'B_cross_page': 0, 'B_start_lowercase': 8}
{'source': 'S04', 'pages': 24, 'skipped_pages': 3, 'A_chunks': 7, 'A_cross_page': 7, 'A_start_lowercase': 5, 'B_chunks': 21, 'B_cross_page': 0, 'B_start_lowercase': 0}
{'source': 'S05', 'pages': 10, 'skipped_pages': 0, 'A_chunks': 4, 'A_cross_page': 4, 'A_start_lowercase': 1, 'B_chunks': 10, 'B_cross_page': 0, 'B_start_lowercase': 0}

Total prepared record

In [ ]:
# Cell 5A2 — Export prepared_text.jsonl and validate (B3 contract)
approach = "page-aware-paragraph-packing"
prepared_path = PROCESSED_DIR / "prepared_text.jsonl"
prepared_path.parent.mkdir(parents=True, exist_ok=True)
with prepared_path.open("w", encoding="utf-8") as out:
    for row in prepared_records:
        out.write(json.dumps(row, ensure_ascii=False) + "\n")

with prepared_path.open("r", encoding="utf-8") as fh:
    exported = [json.loads(l) for l in fh if l.strip()]

assert exported and len(exported) == len(prepared_records), "Record count mismatch"
assert all(r["text"].strip() for r in exported), "Empty chunk found"
assert all(r["source_id"] and r["pdf_page"] for r in exported), "Missing provenance"
assert not any("/gid" in r["text"] for r in exported), "Glyph-code noise left in text"
assert not any(re.search(r"(?m)^www\.utm\.my$", r["text"]) for r in exported), "Footer left in text"
assert set(r["source_id"] for r in exported) == {s["source_id"] for s in SOURCES}, "A source is missing"

print("Prepared file:", prepared_path, "| size:", prepared_path.stat().st_size, "bytes")
print("Records per source:", {sid: sum(r['source_id'] == sid for r in exported) for sid in sorted({r['source_id'] for r in exported})})
print("Currency labels:", sorted({(r['source_id'], r['currency_status']) for r in exported}))
print("Validation: PASS (also read the sample records manually).")

Prepared file: /content/week3_project/data/processed/prepared_text.jsonl | size: 321628 bytes
Records per source: {'S01': 8, 'S02': 176, 'S03': 69, 'S04': 21, 'S05': 10}
Currency labels: [('S01', 'current_2026'), ('S02', 'current_2026'), ('S03', 'currency_not_verified'), ('S04', 'currency_not_verified'), ('S05', 'currency_not_verified')]
Validation: PASS (also read the sample records manually).


In [ ]:
# B3 — basic output check (lab's required lines)
assert prepared_path.exists(), 'Prepared output missing.'
assert prepared_path.stat().st_size > 0, 'Prepared output is empty.'
print('Output:', prepared_path, 'Size:', prepared_path.stat().st_size)
print('Validation: file exists and is non-empty; verify content manually too.')

Output: /content/week3_project/data/processed/prepared_text.jsonl Size: 321628
Validation: file exists and is non-empty; verify content manually too.


In [ ]:
# B4.1 — Register lineage for each source (run in S01..S05 order)
for s in SOURCES:
    register_lineage(
        source=s["path"],
        output=prepared_path,
        processing=f"{approach}; max_chars={s['size']}; footers removed; whitespace normalised; image-only pages skipped"
    )
print("Registered entries:", len(lineage_entries))

Registered: utm_academic_calendar_2026_2027.pdf -> prepared_text.jsonl
Registered: utm_academic_regulations.pdf -> prepared_text.jsonl
Registered: utm_handbook.pdf -> prepared_text.jsonl
Registered: utm_User-Manual_NewStudent_Course-Registration_update-202620271.pdf -> prepared_text.jsonl
Registered: utm_User-Manual_Course-Registration-Senior-Student_update202620271.pdf -> prepared_text.jsonl
Registered entries: 5


In [ ]:
# Cell 6A — Generate common lineage manifest (lab cell + source metadata)
assert lineage_entries, "No lineage entries registered."
lineage_records = []
by_name = {s["path"].name: s for s in SOURCES}
for i, entry in enumerate(lineage_entries.values(), 1):
    source, output, processing = entry["source"], entry["output"], entry["processing"]
    assert source.is_file() and output.is_file() and output.stat().st_size > 0
    cfg = by_name[source.name]
    lineage_records.append({
        "source_id": cfg["source_id"],
        "source_filename": str(source.relative_to(RAW_DIR.resolve())),
        "source_sha256": source_hash(source),
        "prepared_output": str(output.relative_to(BASE_DIR.resolve())),
        "prepared_sha256": source_hash(output),
        "processing_note": processing,
        "source_version": f"{cfg['edition']} ({cfg['doc_date']})",
        "currency_status": cfg["currency"],
        "access_rights": "Public official UTM document; coursework use only; raw file not redistributed",
        "known_limitations": cfg["note"],
        "skipped_pages": cfg["skipped_pages"],
        "status": "Prepared; quality review pending",
    })
manifest = save_manifest(lineage_records)
print("\nCOMMON LINEAGE MANIFEST GENERATED")
print("Total lineage entries:", len(manifest["records"]))

Saved lineage: /content/week3_project/data/week3_manifest.json

COMMON LINEAGE MANIFEST GENERATED
Total lineage entries: 5


In [ ]:
# Cell 6B — Verify Common Lineage Manifest
# REQUIRED: Run after Cell 6A

import json
manifest_path = BASE_DIR / "data" / "week3_manifest.json"
assert manifest_path.is_file(), (
    "Common lineage manifest not found."
)
with manifest_path.open("r", encoding="utf-8") as file:
    saved_manifest = json.load(file)
saved_records = saved_manifest.get("records", [])
assert saved_records, (
    "No lineage records found in saved manifest."
)
assert len(saved_records) == len(lineage_entries), (
    "Saved lineage count does not match registration."
)
for record in saved_records:
    source = RAW_DIR / record["source_filename"]
    output = BASE_DIR / record["prepared_output"]
    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert source_hash(source) == record["source_sha256"], (
        f"Source checksum mismatch: {source}"
    )
    assert source_hash(output) == record["prepared_sha256"], (
        f"Prepared output checksum mismatch: {output}"
    )
print("COMMON LINEAGE VALIDATION: PASS")
print("Verified records:", len(saved_records))
print("Manifest:", manifest_path)


COMMON LINEAGE VALIDATION: PASS
Verified records: 5
Manifest: /content/week3_project/data/week3_manifest.json


In [ ]:
# Cell 7 — REQUIRED: Common V1 pipeline utilities

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PIPE_DIR = BASE_DIR / "data" / "pipeline_v1"
PIPE_DIR.mkdir(parents=True, exist_ok=True)

PIPELINE_RUNS = {}

def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def resolve_source(relative_path, folder=False):
    source = RAW_DIR / relative_path
    if folder:
        if not source.is_dir():
            raise FileNotFoundError(f"Source folder not found: {source}")
    else:
        if not source.is_file():
            raise FileNotFoundError(f"Source file not found: {source}")
    return source

def execute_pipeline(kind, source, inspect_fn, prepare_fn, configuration):
    # INGEST: identify relevant source files
    if kind == "images":
        source = Path(source)
        inputs = sorted(p for p in source.iterdir()
                        if p.is_file() and p.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"})
    elif kind == "documents":
        inputs = [Path(p) for p in source]
    else:
        inputs = [Path(source)]

    if not inputs or any(not p.is_file() for p in inputs):
        raise ValueError("No valid source files to ingest.")

    # INSPECT
    inspection = inspect_fn(inputs, configuration)

    output_dir = PIPE_DIR / kind
    output_dir.mkdir(parents=True, exist_ok=True)

    # PREPARE / STORE
    outputs, mappings, method = prepare_fn(inputs, output_dir, configuration)
    outputs = [Path(p) for p in outputs]

    # VALIDATE OUTPUT
    if not outputs:
        raise ValueError("No prepared outputs were produced.")
    for output in outputs:
        if not output.is_file():
            raise FileNotFoundError(f"Prepared output missing: {output}")
        if output.stat().st_size == 0:
            raise ValueError(f"Prepared output is empty: {output}")

    validation = {
        "output_count": len(outputs),
        "output_sha256": [sha256_file(p) for p in outputs],
        "output_bytes": [p.stat().st_size for p in outputs],
    }

    # RECORD LINEAGE
    record = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "data_type": kind,
        "configuration": configuration,
        "processing_method": method,
        "sources": [{"path": str(p), "sha256": sha256_file(p)} for p in inputs],
        "outputs": [str(p) for p in outputs],
        "source_output_mapping": mappings,
        "inspection": inspection,
        "validation": validation,
        "source_version": configuration.get("source_version", "Not verified"),
        "access_rights": configuration.get("access_rights", "Not verified — check before sharing"),
    }

    manifest_path = output_dir / "pipeline_manifest.json"
    manifest_path.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"SUCCESS: {kind} | Inputs: {len(inputs)} | Outputs: {len(outputs)}")
    print("Manifest:", manifest_path)
    return record

def register_and_run(kind, source, inspect_fn, prepare_fn, configuration):
    def runner():
        return execute_pipeline(kind, source, inspect_fn, prepare_fn, configuration)
    first = runner()
    PIPELINE_RUNS[kind] = {"first": first, "runner": runner}
    return first

print("Cell 7 ready.")
print("Run ONLY the selected Cell 8 pathway(s).")
print("Common pipeline utilities are available.")

Cell 7 ready.
Run ONLY the selected Cell 8 pathway(s).
Common pipeline utilities are available.


In [ ]:
# Cell 8A — CONDITIONAL REQUIRED: PDF / TXT (UniGuide AI: five official UTM PDFs)

!pip -q install pypdf
import re, json
from pypdf import PdfReader

# ---- Step 1: configuration = the SAME decisions made in Part B ----
CONFIG_A = {
    "method": "page-aware-paragraph-packing",
    "min_page_chars": 20,
    "source_version": "See docs/week3_data_inventory.md (S01-S05)",
    "access_rights": "Public official UTM documents; coursework use only; raw files not redistributed",
    "sources": [
        dict(source_id="S01", filename="utm_academic_calendar_2026_2027.pdf",
             language_default="en", language_by_page={1: "en", 2: "ms"},
             doc_date="2026-05-14", edition="Academic Calendar 2026/2027 (corrected version)",
             currency="current_2026", page_offset=0, drop=[r"^www\.utm\.my$"],
             splitter="calendar", size=1200,
             note="Page 2 (Malay) has date errors; treat page 1 (English) as primary."),
        dict(source_id="S02", filename="utm_academic_regulations.pdf",
             language_default="en", language_by_page={},
             doc_date="2026", edition="Academic Regulations, Full-Time UG, 16th Edition",
             currency="current_2026", page_offset=17, drop=[],
             splitter="paragraph", size=800,
             note="PDF page = printed page + 17."),
        dict(source_id="S03", filename="utm_handbook.pdf",
             language_default="en", language_by_page={},
             doc_date="2010", edition="Academic Advising: Ensuring Student Success (2010)",
             currency="currency_not_verified", page_offset=0,
             drop=[r"^\d+ \| Academic Advising: Ensuring Student Success$"],
             splitter="paragraph", size=800,
             note="2010 booklet; describes older systems (AIMS 2000)."),
        dict(source_id="S04",
             filename="utm_User-Manual_NewStudent_Course-Registration_update-202620271.pdf",
             language_default="en", language_by_page={},
             doc_date="2026-10-05", edition="New-student registration manual (update 2026/2027)",
             currency="currency_not_verified", page_offset=0, drop=[r"^www\.utm\.my$"],
             splitter="paragraph", size=1200,
             note="Slide deck; screenshots (image-only) show 2024 sessions."),
        dict(source_id="S05",
             filename="utm_User-Manual_Course-Registration-Senior-Student_update202620271.pdf",
             language_default="en", language_by_page={},
             doc_date="2026-10-06", edition="Senior-student registration manual (update 2026/2027)",
             currency="currency_not_verified", page_offset=0, drop=[r"^www\.utm\.my$"],
             splitter="paragraph", size=1200,
             note="Slide deck; tables and slip are image-only; screenshots show 2024 sessions."),
    ],
}
SOURCE_A = [resolve_source(s["filename"]) for s in CONFIG_A["sources"]]   # stops if a file is missing

# ---- Step 2: helper functions (identical to Part B) ----
def clean_page(text, drop_patterns):
    text = re.sub(r"/gid\d+", "", text)
    keep = [ln.rstrip() for ln in text.splitlines()
            if not any(re.match(p, ln.strip()) for p in drop_patterns)]
    text = "\n".join(keep)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def split_units(text):
    return [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]

def pack(units, size):
    chunks, cur = [], ""
    for u in units:
        if len(u) > size:
            if cur:
                chunks.append(cur); cur = ""
            buf = ""
            for pc in re.split(r"(?<=[.!?;:])\s+|\n", u):
                if len(buf) + len(pc) + 1 <= size:
                    buf += (" " if buf else "") + pc
                else:
                    if buf: chunks.append(buf)
                    buf = pc[:size] if len(pc) > size else pc
            if buf: chunks.append(buf)
        elif len(cur) + len(u) + (2 if cur else 0) <= size:
            cur += ("\n\n" if cur else "") + u
        else:
            chunks.append(cur); cur = u
    if cur:
        chunks.append(cur)
    return chunks

def calendar_chunks(text, size):
    parts = [p.strip() for p in re.split(
        r"(?m)^(?=SEMESTER I \||SEMESTER II \||SHORT SEMESTER \|)", text) if p.strip()]
    return parts if len(parts) > 1 else pack(split_units(text), size)

def parse_printed_page(text):
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    for cand in ([lines[0], lines[-1]] if lines else []):
        if re.fullmatch(r"\d{1,3}", cand):
            return int(cand), cand
    return None, None

def section_of(text):
    for l in [l.strip() for l in text.splitlines() if l.strip()][:3]:
        m = re.search(r"Section\s+(\d+)", l)
        if m:
            return f"Section {m.group(1)}"
    return None

def cfg_for(path, cfg):
    return next(s for s in cfg["sources"] if s["filename"] == Path(path).name)

# ---- Step 3: INSPECT stage ----
def inspect_documents(inputs, cfg):
    report = {}
    for src in inputs:
        s = cfg_for(src, cfg)
        pages = [p.extract_text() or "" for p in PdfReader(str(src)).pages]
        chars = sum(len(t) for t in pages)
        if chars == 0:
            raise ValueError(f"{s['source_id']}: no extractable text (scanned PDF? needs OCR).")
        report[s["source_id"]] = {
            "file": src.name, "pages": len(pages), "characters": chars,
            "low_text_pages": [i + 1 for i, t in enumerate(pages) if len(t.strip()) < cfg["min_page_chars"]],
            "gid_glyph_tokens": sum(len(re.findall(r"/gid\d+", t)) for t in pages),
        }
    return report

# ---- Step 4: PREPARE / STORE stage ----
def prepare_documents(inputs, out, cfg):
    records, mapping = [], []
    target = out / "prepared_text.jsonl"
    for src in inputs:
        s = cfg_for(src, cfg)
        sha = sha256_file(src)
        reader = PdfReader(str(src))
        pages_clean, last_section = [], None
        for pno, page in enumerate(reader.pages, 1):
            raw = page.extract_text() or ""
            printed, printed_line, inferred = None, None, False
            if s["page_offset"]:
                printed, printed_line = parse_printed_page(raw)
                if printed is None and pno >= 18:
                    printed, inferred = pno - s["page_offset"], True
                sec = section_of(raw)
                if sec: last_section = sec
                if printed_line:
                    raw = "\n".join(l for l in raw.splitlines() if l.strip() != printed_line)
                raw = "\n".join(l for l in raw.splitlines()
                                if not re.fullmatch(r"\s*(Academic Regulations\s+)?Section\s+\d+\s*", l))
            text = clean_page(raw, s["drop"])
            if len(text) < cfg["min_page_chars"]:
                continue                                  # image-only / blank page: skipped, listed by INSPECT
            pages_clean.append((pno, text, printed, last_section, inferred))

        n_src = 0
        for pno, text, printed, sec, inferred in pages_clean:
            chunks = (calendar_chunks(text, s["size"]) if s["splitter"] == "calendar" and pno == 1
                      else pack(split_units(text), s["size"]))
            lang = s["language_by_page"].get(pno, s["language_default"])
            for k, ch in enumerate(chunks, 1):
                records.append({
                    "record_id": f"{s['source_id']}-p{pno:03d}-{k:02d}",
                    "source_id": s["source_id"],
                    "source": str(src.relative_to(RAW_DIR)),
                    "source_sha256": sha,
                    "edition": s["edition"],
                    "doc_date": s["doc_date"],
                    "currency_status": s["currency"],
                    "language": lang,
                    "pdf_page": pno,
                    "printed_page": printed,
                    "printed_page_inferred": inferred,
                    "section": sec,
                    "reliability_note": (s["note"] if s["source_id"] != "S01" or pno == 2 else None),
                    "processing": f"page-aware; paragraph/sentence packing; max_chars={s['size']}; footers removed; whitespace normalised",
                    "text": ch,
                })
                n_src += 1
        if n_src == 0:
            raise ValueError(f"{s['source_id']}: no chunks produced.")
        mapping.append({"source_id": s["source_id"], "source": str(src),
                        "output": str(target), "records": n_src})

    with target.open("w", encoding="utf-8") as fh:
        for r in records:
            fh.write(json.dumps(r, ensure_ascii=False) + "\n")

    # stage contract: provenance present on every record, no noise left
    assert all(r["text"].strip() and r["source_id"] and r["pdf_page"] for r in records), "Provenance/text check failed"
    assert not any("/gid" in r["text"] for r in records), "Glyph codes left in text"
    return [target], mapping, f"{cfg['method']} (per-source max_chars; see configuration)"

# ---- Step 5: execute and register ----
run_A = register_and_run("documents", SOURCE_A, inspect_documents, prepare_documents, CONFIG_A)

# ---- Step 6: evidence + comparison with Part B ----
print("\nInspection:")
for sid, info in run_A["inspection"].items():
    print(" ", sid, info)
print("\nSource -> output mapping:")
for m in run_A["source_output_mapping"]:
    print(" ", m["source_id"], "->", Path(m["output"]).name, "| records:", m["records"])

partB_path = BASE_DIR / "data" / "processed" / "prepared_text.jsonl"
partC_path = Path(run_A["outputs"][0])
if partB_path.is_file():
    same = sha256_file(partB_path) == sha256_file(partC_path)
    print("\nPart C output identical to Part B output:", same)
    if not same:
        print("  Differences must be explained in docs/week3_pipeline_decisions.md.")
sample = json.loads(next(l for l in partC_path.open(encoding="utf-8") if '"S02-p040' in l))
print("\nRepresentative record:", json.dumps({k: sample[k] for k in
      ("record_id", "pdf_page", "printed_page", "section", "currency_status")}), "|", sample["text"][:150])
print("Cell 8A completed successfully.")

SUCCESS: documents | Inputs: 5 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents/pipeline_manifest.json

Inspection:
  S01 {'file': 'utm_academic_calendar_2026_2027.pdf', 'pages': 2, 'characters': 8335, 'low_text_pages': [], 'gid_glyph_tokens': 0}
  S02 {'file': 'utm_academic_regulations.pdf', 'pages': 81, 'characters': 105131, 'low_text_pages': [1, 2, 3, 11, 13, 15, 81], 'gid_glyph_tokens': 75}
  S03 {'file': 'utm_handbook.pdf', 'pages': 28, 'characters': 39859, 'low_text_pages': [1, 28], 'gid_glyph_tokens': 0}
  S04 {'file': 'utm_User-Manual_NewStudent_Course-Registration_update-202620271.pdf', 'pages': 24, 'characters': 3752, 'low_text_pages': [1, 2, 5], 'gid_glyph_tokens': 0}
  S05 {'file': 'utm_User-Manual_Course-Registration-Senior-Student_update202620271.pdf', 'pages': 10, 'characters': 2048, 'low_text_pages': [], 'gid_glyph_tokens': 0}

Source -> output mapping:
  S01 -> prepared_text.jsonl | records: 8
  S02 -> prepared_text.jsonl | records: 176
  S03 ->

In [ ]:
# Cell 9 — REQUIRED: Repeatability and provenance verification
assert PIPELINE_RUNS, "No pipeline runs registered. Run at least one selected Cell 8 pathway first."
print("--- Repeatability Verification ---")
for kind, entry in PIPELINE_RUNS.items():
    first = entry["first"]
    second = entry["runner"]()
    assert first["sources"] == second["sources"], f"Source fingerprint changed for {kind}"
    assert first["validation"] == second["validation"], f"Prepared output changed for {kind}"
    assert first["source_output_mapping"] == second["source_output_mapping"], \
        f"Source-to-output mapping changed for {kind}"
    print(f"PASS: {kind} — source fingerprints, prepared outputs and mappings match")
print("\nAll selected pipelines passed repeatability smoke checks.")

--- Repeatability Verification ---
SUCCESS: documents | Inputs: 5 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents/pipeline_manifest.json
PASS: documents — source fingerprints, prepared outputs and mappings match

All selected pipelines passed repeatability smoke checks.
